# 01 — Patch CSV Data Fixes

This notebook contains only the source-data correction layer.

It takes the raw CSV files, copies them to a patched working folder, applies the same correction logic from the original data-preparation notebook, and writes corrected CSVs.

It does **not** generate report payloads and it does **not** create these deprecated files:

- `disclosures.csv`
- `data_requirements.csv`
- `disclosure_data_map.csv`

Run this notebook first, then run the data-preparation notebook.

In [3]:
from pathlib import Path
import json
import shutil
import zipfile
import pandas as pd
import numpy as np

# ============================================================
# Configuration
# ============================================================

# Keep this as None for automatic detection.
# If your notebook is opened from Downloads, set it manually, for example:
PROJECT_ROOT_OVERRIDE = Path(r"C:\Users\HP\Documents\IFRS_Reporting")

CURRENT_DIR = Path.cwd()

if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE)
else:
    PROJECT_ROOT = None
    for candidate in [CURRENT_DIR, *CURRENT_DIR.parents]:
        if (candidate / "notebooks" / "gen_data").exists() or (candidate / "data.zip").exists() or (candidate / "gen_data").exists():
            PROJECT_ROOT = candidate
            break
    if PROJECT_ROOT is None:
        PROJECT_ROOT = CURRENT_DIR

ZIP_PATH = PROJECT_ROOT / "data.zip"

# Preferred project layout
RAW_DATA_DIR = PROJECT_ROOT / "notebooks" / "gen_data" / "csv"
PATCHED_DATA_DIR = PROJECT_ROOT / "notebooks" / "gen_data" / "csv_patched"

# Fallback layouts for standalone notebook runs
if not RAW_DATA_DIR.exists():
    for candidate in [
        PROJECT_ROOT / "gen_data",
        PROJECT_ROOT / "data",
        PROJECT_ROOT / "csv",
    ]:
        if candidate.exists() and (candidate / "banks.csv").exists():
            RAW_DATA_DIR = candidate
            PATCHED_DATA_DIR = candidate.parent / "csv_patched"
            break

PATCHED_DATA_DIR.mkdir(parents=True, exist_ok=True)
PATCH_AUDIT_CSV = PATCHED_DATA_DIR / "patch_audit_log.csv"
PATCH_SUMMARY_JSON = PATCHED_DATA_DIR / "patch_summary.json"

REPORTING_YEAR = 2024
COMPARATIVE_YEARS = [2022, 2023]
FLEET_EMISSION_FACTOR_KGCO2_PER_LITRE = 2.67

EXCLUDED_CSV_FILES = {
    "disclosures.csv",
    "data_requirements.csv",
    "disclosure_data_map.csv",
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("PATCHED_DATA_DIR:", PATCHED_DATA_DIR)
print("ZIP_PATH:", ZIP_PATH)


PROJECT_ROOT: C:\Users\HP\Documents\IFRS_Reporting
RAW_DATA_DIR: C:\Users\HP\Documents\IFRS_Reporting\notebooks\gen_data\csv
PATCHED_DATA_DIR: C:\Users\HP\Documents\IFRS_Reporting\notebooks\gen_data\csv_patched
ZIP_PATH: C:\Users\HP\Documents\IFRS_Reporting\data.zip


In [4]:
# ============================================================
# 1. Input discovery and loading
# ============================================================

if not RAW_DATA_DIR.exists() and ZIP_PATH.exists():
    print(f"Extracting {ZIP_PATH} ...")
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(RAW_DATA_DIR)

if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(
        f"Could not find CSV folder: {RAW_DATA_DIR}. Put source CSVs in notebooks/gen_data/csv, gen_data, data, or place data.zip next to the project."
    )

# If a ZIP archive extracted into an inner data/ folder, use that folder.
if not (RAW_DATA_DIR / "banks.csv").exists() and (RAW_DATA_DIR / "data" / "banks.csv").exists():
    RAW_DATA_DIR = RAW_DATA_DIR / "data"
    csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {RAW_DATA_DIR}")

PATCHED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Clear previous patched CSVs first to avoid stale/deprecated files.
for existing in PATCHED_DATA_DIR.glob("*.csv"):
    existing.unlink()

for src_file in csv_files:
    if src_file.name in EXCLUDED_CSV_FILES:
        continue
    shutil.copy2(src_file, PATCHED_DATA_DIR / src_file.name)

for deprecated_name in EXCLUDED_CSV_FILES:
    deprecated_path = PATCHED_DATA_DIR / deprecated_name
    if deprecated_path.exists():
        deprecated_path.unlink()

def load_tables(folder: Path) -> dict:
    return {p.stem: pd.read_csv(p) for p in sorted(folder.glob("*.csv"))}

tables = load_tables(PATCHED_DATA_DIR)

print(f"Copied and loaded {len(tables)} tables.")
for name, df in sorted(tables.items()):
    print(f"  {name:32s} {df.shape[0]:>6} rows x {df.shape[1]:>3} cols")


# ============================================================
# 2. Audit helpers
# ============================================================

patch_log = []

def normalise_audit_value(value):
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, np.generic):
        return value.item()
    return value

def safe_float(value):
    try:
        value = float(value)
        return value if value == value else None
    except Exception:
        return None

def values_differ(old_value, new_value, tol=1e-9):
    old = normalise_audit_value(old_value)
    new = normalise_audit_value(new_value)
    if old is None and new is None:
        return False
    try:
        return abs(float(old) - float(new)) > tol
    except Exception:
        return old != new

def record_fix(bank_id, table, field, old_value, new_value, reason, row_id=None):
    if not values_differ(old_value, new_value):
        return
    patch_log.append({
        "bank_id": bank_id,
        "table": table,
        "field": field,
        "row_id": row_id,
        "old_value": normalise_audit_value(old_value),
        "new_value": normalise_audit_value(new_value),
        "reason": reason,
    })


# ============================================================
# 3. Helper: sector exposure KPIs used by scenario fixes
# ============================================================

HIGH_CARBON_NACE = {"B06", "B07", "B08", "B09", "C19", "C20", "D35", "H49", "H50", "H51", "C24"}
FOSSIL_FUEL_NACE = {"B06", "B07", "C19", "D35"}

def is_high_carbon_nace(code):
    code = str(code or "").upper()
    return code in HIGH_CARBON_NACE

def is_fossil_fuel_nace(code):
    code = str(code or "").upper()
    return code in FOSSIL_FUEL_NACE

def build_sector_kpis(tables):
    if "exposures" not in tables or "counterparties" not in tables:
        return {}

    exp = tables["exposures"].copy()
    cp = tables["counterparties"][["counterparty_id", "nace_code"]].copy()
    merged = exp.merge(cp, on="counterparty_id", how="left")

    sector_kpis = {}
    for bank_id, group in merged.groupby("bank_id"):
        total = group["outstanding_amount_meur"].sum()
        high = group[group["nace_code"].apply(is_high_carbon_nace)]["outstanding_amount_meur"].sum()
        fossil = group[group["nace_code"].apply(is_fossil_fuel_nace)]["outstanding_amount_meur"].sum()

        sector_kpis[bank_id] = {
            "total_loans_meur": round(float(total), 2),
            "high_carbon_sector_exposure_meur": round(float(high), 2),
            "fossil_fuel_exposure_meur": round(float(fossil), 2),
            "high_carbon_sector_exposure_pct": round(float(high / total * 100), 2) if total else None,
            "fossil_fuel_exposure_pct": round(float(fossil / total * 100), 2) if total else None,
        }

    return sector_kpis

sector_kpis = build_sector_kpis(tables)


# ============================================================
# 4. Patch vehicles.csv — Scope 1 fleet emissions
# ============================================================

if "vehicles" in tables:
    veh = tables["vehicles"].copy()

    if "scope1_tco2e_2024_original" not in veh.columns:
        veh["scope1_tco2e_2024_original"] = veh["scope1_tco2e_2024"]

    def recalc_fleet_scope1(row):
        fuel_type = str(row.get("fuel_type", "")).lower()
        fuel_litres = safe_float(row.get("annual_fuel_consumption_l")) or 0.0

        if fuel_type == "electric":
            return 0.0
        if fuel_litres <= 0:
            return 0.0
        return round(fuel_litres * FLEET_EMISSION_FACTOR_KGCO2_PER_LITRE / 1000, 6)

    clean_values = veh.apply(recalc_fleet_scope1, axis=1)

    for idx, new_value in clean_values.items():
        old_value = veh.at[idx, "scope1_tco2e_2024"]
        record_fix(
            veh.at[idx, "bank_id"],
            "vehicles",
            "scope1_tco2e_2024",
            old_value,
            new_value,
            "Recalculated Scope 1 fleet emissions from annual_fuel_consumption_l * 2.67 kgCO2/litre / 1000; electric vehicles set to zero.",
            row_id=veh.at[idx, "vehicle_id"] if "vehicle_id" in veh.columns else idx,
        )

    veh["scope1_tco2e_2024_clean"] = clean_values
    veh["scope1_tco2e_2024"] = clean_values

    tables["vehicles"] = veh
    veh.to_csv(PATCHED_DATA_DIR / "vehicles.csv", index=False)
    print("Vehicle patch complete.")


# ============================================================
# 5. Patch banks.csv — align master fields to 2024 financial_summary
# ============================================================

if "banks" in tables and "financial_summary" in tables:
    banks = tables["banks"].copy()
    fs = tables["financial_summary"].copy()

    for idx, bank in banks.iterrows():
        bank_id = bank["bank_id"]
        fs24 = fs[(fs["bank_id"] == bank_id) & (fs["reporting_year"] == REPORTING_YEAR)]
        if fs24.empty:
            continue

        fs24 = fs24.iloc[0]
        for col in ["total_assets_meur", "total_loans_meur", "tier1_capital_meur", "cet1_ratio_pct"]:
            if col in banks.columns and col in fs24.index:
                old_value = banks.at[idx, col]
                new_value = fs24[col]
                if values_differ(old_value, new_value):
                    banks.at[idx, col] = new_value
                    record_fix(bank_id, "banks", col, old_value, new_value, "Aligned bank master field to 2024 financial_summary.", row_id=bank_id)

    tables["banks"] = banks
    banks.to_csv(PATCHED_DATA_DIR / "banks.csv", index=False)
    print("Bank master patch complete.")


# ============================================================
# 6. Patch governance.csv — director percentages and board agenda
# ============================================================

if "governance" in tables:
    gov = tables["governance"].copy()

    if "board_size" in gov.columns:
        for idx, row in gov.iterrows():
            board_size = safe_float(row.get("board_size"))
            if not board_size:
                continue

            for col in ["independent_directors_pct", "board_climate_expertise_pct"]:
                if col not in gov.columns:
                    continue
                old_value = safe_float(row.get(col))
                if old_value is None:
                    continue

                director_count = round(old_value / 100 * board_size)
                new_value = round(director_count / board_size * 100, 1)

                if values_differ(old_value, new_value):
                    gov.at[idx, col] = new_value
                    record_fix(row.get("bank_id"), "governance", col, old_value, new_value, "Rounded percentage to a whole-number director count.", row_id=f"{row.get('bank_id')}-{row.get('reporting_year')}")

    bm = tables.get("board_minutes_extract")
    required_cols = {"bank_id", "reporting_year", "committee_type", "climate_agenda_flag"}
    if bm is not None and required_cols.issubset(bm.columns) and "climate_on_board_agenda_pct" in gov.columns:
        full_board = bm[bm["committee_type"].astype(str).str.lower().eq("full_board")].copy()
        agenda = full_board.groupby(["bank_id", "reporting_year"])["climate_agenda_flag"].mean().mul(100).round(1).reset_index()

        for idx, row in gov.iterrows():
            match = agenda[(agenda["bank_id"] == row["bank_id"]) & (agenda["reporting_year"] == row["reporting_year"])]
            if not match.empty:
                old_value = safe_float(row.get("climate_on_board_agenda_pct"))
                new_value = float(match.iloc[0]["climate_agenda_flag"])
                gov.at[idx, "climate_on_board_agenda_pct"] = new_value
                record_fix(row["bank_id"], "governance", "climate_on_board_agenda_pct", old_value, new_value, "Recomputed from Full Board minutes climate_agenda_flag.", row_id=f"{row.get('bank_id')}-{row.get('reporting_year')}")

    tables["governance"] = gov
    gov.to_csv(PATCHED_DATA_DIR / "governance.csv", index=False)
    print("Governance patch complete.")


# ============================================================
# 7. Patch climate_scenarios.csv — methodology and magnitude
# ============================================================

METHODOLOGY_MAP = {
    "Net Zero 2050": "NGFS Net Zero 2050 scenario applied to the banking book using sector-pathway parameters, carbon-price assumptions and counterparty sector exposures.",
    "Delayed Transition": "NGFS Delayed Transition scenario applied through transition-risk shocks, carbon-price sensitivity and delayed policy response assumptions.",
    "Current Policies": "NGFS Current Policies scenario applied using higher physical-risk assumptions and limited additional transition-policy tightening.",
    "Below 2": "Orderly transition scenario aligned with a below-2°C pathway, using sector-level decarbonisation and exposure sensitivity analysis.",
}

def scenario_methodology(name):
    name = str(name)
    for key, value in METHODOLOGY_MAP.items():
        if key.lower() in name.lower():
            return value
    return "Climate scenario analysis applied using portfolio exposure, sector sensitivity and financial-impact assumptions."

def scenario_resilience(row):
    scenario_type = str(row.get("scenario_type", "")).lower()
    high_risk = safe_float(row.get("high_risk_exposure_pct")) or 0
    if "orderly" in scenario_type or high_risk < 10:
        return "The portfolio shows comparatively higher resilience under this scenario, with manageable exposure concentrations."
    if "disorderly" in scenario_type or high_risk < 25:
        return "The portfolio remains exposed to transition and physical risk hotspots, requiring active risk mitigation and client engagement."
    return "The scenario indicates elevated vulnerability and requires strengthened mitigation actions, portfolio steering and resilience planning."

if "climate_scenarios" in tables:
    cs = tables["climate_scenarios"].copy()

    if "methodology_notes" not in cs.columns:
        cs["methodology_notes"] = None
    if "resilience_assessment" not in cs.columns:
        cs["resilience_assessment"] = None

    for idx, row in cs.iterrows():
        if pd.isna(row.get("methodology_notes")) or not str(row.get("methodology_notes")).strip():
            new_value = scenario_methodology(row.get("scenario_name"))
            cs.at[idx, "methodology_notes"] = new_value
            record_fix(row.get("bank_id"), "climate_scenarios", "methodology_notes", None, new_value, "Added scenario methodology notes.", row_id=row.get("scenario_id"))

        if pd.isna(row.get("resilience_assessment")) or not str(row.get("resilience_assessment")).strip():
            new_value = scenario_resilience(row)
            cs.at[idx, "resilience_assessment"] = new_value
            record_fix(row.get("bank_id"), "climate_scenarios", "resilience_assessment", None, new_value, "Added scenario resilience assessment.", row_id=row.get("scenario_id"))

        bank_id = row.get("bank_id")
        fossil_exposure = sector_kpis.get(bank_id, {}).get("fossil_fuel_exposure_meur")
        high_risk_pct = safe_float(row.get("high_risk_exposure_pct"))

        if fossil_exposure is not None and high_risk_pct is not None and "stranded_assets_estimate_meur" in cs.columns:
            old_value = safe_float(row.get("stranded_assets_estimate_meur"))
            new_value = round(fossil_exposure * high_risk_pct / 100, 2)
            if old_value is not None and values_differ(old_value, new_value, tol=0.01):
                cs.at[idx, "stranded_assets_estimate_meur"] = new_value
                cs.at[idx, "scenario_magnitude_adjusted_flag"] = True
                record_fix(bank_id, "climate_scenarios", "stranded_assets_estimate_meur", old_value, new_value, "Recalculated as fossil-fuel exposure multiplied by high-risk exposure percentage.", row_id=row.get("scenario_id"))

        if "revenue_at_risk_meur" in cs.columns and "financial_summary" in tables:
            fs = tables["financial_summary"]
            fs24 = fs[(fs["bank_id"] == bank_id) & (fs["reporting_year"] == REPORTING_YEAR)]
            if not fs24.empty:
                revenue = safe_float(fs24.iloc[0].get("total_revenue_meur"))
                old_value = safe_float(row.get("revenue_at_risk_meur"))
                if revenue is not None and old_value is not None:
                    cap = round(revenue * 0.30, 2)
                    new_value = min(old_value, cap)
                    if values_differ(old_value, new_value, tol=0.01):
                        cs.at[idx, "revenue_at_risk_meur"] = new_value
                        cs.at[idx, "scenario_magnitude_adjusted_flag"] = True
                        record_fix(bank_id, "climate_scenarios", "revenue_at_risk_meur", old_value, new_value, "Capped at 30% of 2024 revenue to avoid impossible report magnitudes.", row_id=row.get("scenario_id"))

    tables["climate_scenarios"] = cs
    cs.to_csv(PATCHED_DATA_DIR / "climate_scenarios.csv", index=False)
    print("Climate scenario patch complete.")


# ============================================================
# 8. Patch climate_risk_register.csv — mitigation and ratings
# ============================================================

def aligned_mitigation(risk_name, risk_category):
    name = str(risk_name or "").lower()
    category = str(risk_category or "").lower()

    if "flood" in name:
        return "Collateral revaluation, flood-zone mapping overlay and updated mortgage underwriting criteria"
    if "wildfire" in name:
        return "Wildfire hazard mapping, collateral revaluation and borrower resilience engagement"
    if "heat" in name or "agric" in name:
        return "Climate scenario stress testing, sector monitoring and borrower adaptation engagement"
    if "stranded" in name or "fossil" in name:
        return "Sector exposure limits, enhanced due diligence and portfolio decarbonisation glide-path monitoring"
    if "carbon pricing" in name or "policy" in category:
        return "Sector exposure limits, carbon-cost sensitivity analysis and enhanced due diligence"
    if "ev shift" in name or "technology" in category:
        return "Technology transition monitoring, sector concentration limits and client transition-plan engagement"
    if "reputational" in category or "scrutiny" in name:
        return "Enhanced financed-emissions disclosure controls and client engagement on transition plans"
    return None

def risk_rating_from_scores(likelihood, severity):
    likelihood = safe_float(likelihood) or 0
    severity = safe_float(severity) or 0
    score = likelihood * severity
    if score >= 15:
        return "critical"
    if score >= 8:
        return "high"
    if score >= 3:
        return "medium"
    return "low"

if "climate_risk_register" in tables:
    rr = tables["climate_risk_register"].copy()

    for idx, row in rr.iterrows():
        new_action = aligned_mitigation(row.get("risk_name"), row.get("risk_category"))
        if new_action:
            old_action = row.get("mitigation_actions")
            if old_action != new_action:
                rr.at[idx, "mitigation_actions"] = new_action
                record_fix(row.get("bank_id"), "climate_risk_register", "mitigation_actions", old_action, new_action, "Aligned mitigation action to risk type.", row_id=row.get("risk_id"))

        new_rating = risk_rating_from_scores(row.get("likelihood_score"), row.get("severity_score"))
        old_rating = row.get("risk_rating")
        if old_rating != new_rating:
            rr.at[idx, "risk_rating"] = new_rating
            record_fix(row.get("bank_id"), "climate_risk_register", "risk_rating", old_rating, new_rating, "Recalculated from likelihood_score * severity_score using the stated 5x5 risk matrix.", row_id=row.get("risk_id"))

    tables["climate_risk_register"] = rr
    rr.to_csv(PATCHED_DATA_DIR / "climate_risk_register.csv", index=False)
    print("Climate risk register patch complete.")


# ============================================================
# 9. Create climate_opportunities.csv only if missing
# ============================================================

if "climate_opportunities" not in tables:
    # 2.4.2 Create climate_opportunities.csv
    # Fills the IFRS S2 §9 opportunity disclosure gap.
    # Each bank gets 5 opportunities reflecting its archetype.
    # The Writer Agent uses this so it does NOT invent opportunity narrative.

    BANK_ARCHETYPES = {
        "BANK01": "large_universal",
        "BANK02": "large_universal",
        "BANK03": "mid_size_commercial",
        "BANK04": "specialized_green",
        "BANK05": "corporate_laggard",
    }

    # Template: (opportunity_type, category, description, revenue_impact_meur, time_horizon, ifrs_s2_ref)
    # Values are calibrated to bank size — BANK01 has 850B assets, BANK04 18B
    OPPORTUNITY_TEMPLATES = {
        "large_universal": [
            ("green_loan_growth",
             "Products_and_services",
             "Expansion of green and sustainability-linked loan book aligned with EU Taxonomy, driven by corporate client transition financing demand.",
             180.0, "medium_term", "S2.9"),
            ("sustainable_bond_underwriting",
             "Products_and_services",
             "Growth in green bond and social bond underwriting and distribution fees as sovereign and corporate issuers accelerate ESG capital market activity.",
             95.0, "short_term", "S2.9"),
            ("transition_advisory",
             "Products_and_services",
             "Fee income from climate risk advisory, transition planning support, and ESG due diligence services offered to large corporate clients.",
             42.0, "medium_term", "S2.9"),
            ("renewable_energy_project_finance",
             "Products_and_services",
             "Project finance for solar, wind, and battery storage infrastructure aligned with REPowerEU and national renewable energy targets.",
             130.0, "long_term", "S2.9"),
            ("operational_energy_efficiency",
             "Resource_efficiency",
             "Reduction in energy costs from facility EPC upgrades and REC procurement, lowering Scope 2 emissions and operational expenditure.",
             18.0, "short_term", "S2.9"),
        ],
        "mid_size_commercial": [
            ("green_loan_growth",
             "Products_and_services",
             "Growth of green SME lending under LMA Green Loan Principles, targeting energy-efficient commercial real estate and manufacturing retrofits.",
             55.0, "medium_term", "S2.9"),
            ("sustainable_bond_issuance",
             "Products_and_services",
             "Self-issuance of green bonds to fund green loan portfolio, reducing funding cost and improving ESG investor access.",
             28.0, "medium_term", "S2.9"),
            ("transition_advisory",
             "Products_and_services",
             "Climate transition advisory for mid-market industrial clients subject to ETS obligations and CBAM.",
             14.0, "short_term", "S2.9"),
            ("renewable_energy_project_finance",
             "Products_and_services",
             "Project finance for regional renewable energy cooperatives and municipal solar projects.",
             40.0, "long_term", "S2.9"),
            ("operational_energy_efficiency",
             "Resource_efficiency",
             "Energy cost savings from facility LED and HVAC upgrades across branch network, supported by EPC rating improvements.",
             6.0, "short_term", "S2.9"),
        ],
        "specialized_green": [
            ("green_loan_growth",
             "Products_and_services",
             "Core business expansion: EU Taxonomy-aligned green loans now represent 47% of loan book with strong pipeline growth from SFDR Article 9 fund clients.",
             210.0, "short_term", "S2.9"),
            ("impact_investing_products",
             "Products_and_services",
             "Launch of biodiversity-linked and blue economy financing products addressing emerging EU Nature Restoration Law investment needs.",
             65.0, "medium_term", "S2.9"),
            ("transition_advisory",
             "Products_and_services",
             "Premium ESG transition advisory revenue from corporate and institutional clients benchmarking against SBTi pathways.",
             30.0, "short_term", "S2.9"),
            ("renewable_energy_project_finance",
             "Products_and_services",
             "Offshore wind and green hydrogen project finance in the Netherlands and Nordic markets, aligned with REPowerEU hydrogen strategy.",
             90.0, "long_term", "S2.9"),
            ("operational_energy_efficiency",
             "Resource_efficiency",
             "Near-zero operational emissions profile maintained through 100% renewable energy procurement; cost advantage over peers with higher Scope 2 exposure.",
             12.0, "short_term", "S2.9"),
        ],
        "corporate_laggard": [
            ("green_loan_growth",
             "Products_and_services",
             "Early-stage green loan product launch targeting Iberian SME market; green loan share currently below 2% with significant growth headroom.",
             12.0, "long_term", "S2.9"),
            ("sustainable_bond_issuance",
             "Products_and_services",
             "Planned issuance of first sustainability bond to access ESG investor base and reduce funding cost premium currently paid versus green peers.",
             8.0, "long_term", "S2.9"),
            ("transition_advisory",
             "Products_and_services",
             "Basic climate risk advisory offering under development; planned launch for large corporate segment from 2026.",
             4.0, "long_term", "S2.9"),
            ("renewable_energy_project_finance",
             "Products_and_services",
             "Participation in Iberian solar project finance syndications alongside larger arranger banks, building renewable energy expertise.",
             18.0, "long_term", "S2.9"),
            ("operational_energy_efficiency",
             "Resource_efficiency",
             "Energy audit program initiated across branch network; expected operational cost reduction upon completion of EPC improvement programme.",
             3.0, "medium_term", "S2.9"),
        ],
    }

    rows = []
    opp_counter = 1
    for bank_id, archetype in BANK_ARCHETYPES.items():
        for i, (opp_type, category, description, revenue_impact, time_horizon, ifrs_ref) in \
                enumerate(OPPORTUNITY_TEMPLATES[archetype], start=1):
            rows.append({
                "opportunity_id":              f"OPP-{bank_id}-{i:02d}",
                "bank_id":                     bank_id,
                "reporting_year":              2024,
                "opportunity_type":            opp_type,
                "category":                    category,
                "description":                 description,
                "estimated_revenue_impact_meur": revenue_impact,
                "time_horizon":                time_horizon,
                "confidence_level":            "medium" if archetype != "corporate_laggard" else "low",
                "ifrs_s2_para_evidence":       ifrs_ref,
                "data_source":                 "bank_strategy_documents",
                "linked_risk_category":        (
                    "transition_policy" if "green" in opp_type or "advisory" in opp_type
                    else "physical_acute" if "efficiency" in opp_type
                    else "transition_market"
                ),
            })
            opp_counter += 1

    climate_opportunities = pd.DataFrame(rows)

    # Validation
    assert len(climate_opportunities) == 25, \
        f"Expected 25 rows (5 per bank × 5 banks), got {len(climate_opportunities)}"
    assert climate_opportunities["opportunity_id"].duplicated().sum() == 0, \
        "Duplicate opportunity IDs"
    assert climate_opportunities["bank_id"].isin(tables["banks"]["bank_id"]).all(), \
        "Unknown bank_id in opportunities"

    print(f"climate_opportunities: {len(climate_opportunities)} rows")
    print(climate_opportunities.groupby(["bank_id", "opportunity_type"])["estimated_revenue_impact_meur"].first().to_string())

    climate_opportunities.to_csv(PATCHED_DATA_DIR / "climate_opportunities.csv", index=False)
    print(f"\n{PATCHED_DATA_DIR / 'climate_opportunities.csv'} saved")

    # Load into tables dict
    tables["climate_opportunities"] = climate_opportunities
else:
    print("climate_opportunities.csv already present; kept existing file.")

# ============================================================
# 10. Save patch audit
# ============================================================

patch_df = pd.DataFrame(patch_log)

if not patch_df.empty:
    patch_df.to_csv(PATCH_AUDIT_CSV, index=False)
    print(f"Patch audit saved to: {PATCH_AUDIT_CSV.resolve()}")
    display(patch_df.head(30))
else:
    print("No patch corrections were required.")

summary = {
    "patched_data_dir": str(PATCHED_DATA_DIR.resolve()),
    "total_fixes": int(len(patch_df)),
    "fixes_by_table": patch_df.groupby("table").size().to_dict() if not patch_df.empty else {},
    "removed_generation": [
        "disclosures.csv",
        "data_requirements.csv",
        "disclosure_data_map.csv",
    ],
    "notes": [
        "This notebook does not generate IFRS disclosure catalog CSVs.",
        "Missing data is kept as an internal preparation issue only.",
        "Patched CSVs are ready for the data-understanding, emissions and payload-generation notebook.",
    ],
}

PATCH_SUMMARY_JSON.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))

Copied and loaded 30 tables.
  banks                                 5 rows x  18 cols
  board_minutes_extract               154 rows x  13 cols
  carbon_credits                       28 rows x  19 cols
  climate_financial_effects            35 rows x  14 cols
  climate_opportunities                25 rows x  12 cols
  climate_risk_register                80 rows x  17 cols
  climate_scenarios                    84 rows x  22 cols
  collateral                          228 rows x  19 cols
  counterparties                      305 rows x  20 cols
  counterparty_emissions              900 rows x  12 cols
  employees                            15 rows x  11 cols
  exposures                           603 rows x  20 cols
  facilities                           80 rows x  10 cols
  financial_summary                    15 rows x  17 cols
  ghg_methodology                      20 rows x  13 cols
  governance                           15 rows x  23 cols
  internal_carbon_price                15 r

,bank_id,table,field,row_id,old_value,new_value,reason
0,BANK01,vehicles,scope1_tco2e_2024,VEH0001,4.8674,4.86741,Recalculated Scope 1 fleet emissions from annu...
1,BANK01,vehicles,scope1_tco2e_2024,VEH0002,3.6851,3.685134,Recalculated Scope 1 fleet emissions from annu...
2,BANK01,vehicles,scope1_tco2e_2024,VEH0003,2.6833,2.68335,Recalculated Scope 1 fleet emissions from annu...
3,BANK01,vehicles,scope1_tco2e_2024,VEH0004,0.0708,0.0,Recalculated Scope 1 fleet emissions from annu...
4,BANK01,vehicles,scope1_tco2e_2024,VEH0005,4.869,4.869012,Recalculated Scope 1 fleet emissions from annu...
5,BANK01,vehicles,scope1_tco2e_2024,VEH0006,0.0213,0.0,Recalculated Scope 1 fleet emissions from annu...
6,BANK01,vehicles,scope1_tco2e_2024,VEH0007,4.0712,4.071216,Recalculated Scope 1 fleet emissions from annu...
7,BANK01,vehicles,scope1_tco2e_2024,VEH0008,1.8749,1.874874,Recalculated Scope 1 fleet emissions from annu...
8,BANK01,vehicles,scope1_tco2e_2024,VEH0009,7.9766,7.976625,Recalculated Scope 1 fleet emissions from annu...
9,BANK01,vehicles,scope1_tco2e_2024,VEH0010,3.3925,3.392502,Recalculated Scope 1 fleet emissions from annu...


{
  "patched_data_dir": "C:\\Users\\HP\\Documents\\IFRS_Reporting\\notebooks\\gen_data\\csv_patched",
  "total_fixes": 701,
  "fixes_by_table": {
    "banks": 10,
    "climate_risk_register": 150,
    "climate_scenarios": 95,
    "governance": 8,
    "vehicles": 438
  },
  "removed_generation": [
    "disclosures.csv",
    "data_requirements.csv",
    "disclosure_data_map.csv"
  ],
  "notes": [
    "This notebook does not generate IFRS disclosure catalog CSVs.",
    "Missing data is kept as an internal preparation issue only.",
    "Patched CSVs are ready for the data-understanding, emissions and payload-generation notebook."
  ]
}
